In [0]:
%pip install -q databricks-langchain "langgraph>=0.3,<0.4" databricks-agents "psycopg[binary]>=3.1.0" pydantic databricks-ai-search

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


# Fraud Triage Copilot Agent
**Step 5:** Fraud Triage Copilot (Layer 4) + Lakebase FGAC

- FM endpoint: `databricks-claude-sonnet-5-5`
- Tools (since 2026-10-21 FGAC hardening): all agent-side and Postgres-governed via `LakebaseCases(pg_role='fraud_triage_agent')`: `get_transaction_risk` (calibrated confidence), `explain_top_signals`, `get_customer_history`, `create_fraud_case`. The UC function versions are deprecated and not registered.
- Registered: `workshop_guy_catalog.fe_bar_a0.fraud_triage_agent` v1 @champion
- Experiment: `/Users/alex.owen@databricks.com/fe_bar_fraud_triage_agent`

In [0]:
import sys, json, time
sys.path.insert(0, "/Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo")

import mlflow
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Users/alex.owen@databricks.com/fe_bar_fraud_triage_agent")

from fraud_triage_agent import FraudTriageAgent
from mlflow.types.responses import ResponsesAgentRequest

agent = FraudTriageAgent()
print("Agent ready.")

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc
/local_disk0/.ephemeral_nfs/envs/pythonEnv-8d32adad-a55b-4953-ba99-1953b044617b/lib/python3.12/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
2026/11/01 19:07:41 WARNING mlflow.utils.autologging_utils: MLflow langchain autologging is known to be compatible with 1.0.0 <= langchain, but the installed version is 0.3.30. If you encounter errors during autologging, try upgrading / downgrading langchain to a compatible version, or 

Agent ready.


In [0]:
# E.1: Why was TXN-0234209 flagged?
request = ResponsesAgentRequest(
    input=[{"role": "user", "content": "Why was TXN-0234209 flagged and what should we do?"}]
)
t0 = time.time()
response = agent.predict(request)
print(f"Latency: {time.time()-t0:.1f}s")
print(response.output[0].content[0]["text"])

# FGAC hardening Item 5 VERIFY: trace span table for this run.
trace_id = mlflow.get_last_active_trace_id()
trace = mlflow.get_trace(trace_id)
print(f"\n=== Trace span table  trace_id={trace_id} ===")
rows = []
for s in trace.data.spans:
    if not str(s.span_type).upper().endswith("TOOL"):
        continue
    out = s.outputs
    if hasattr(out, "content"): out = out.content
    if isinstance(out, dict) and "content" in out: out = out["content"]
    try:
        payload = json.loads(out) if isinstance(out, str) else {}
    except Exception:
        payload = {}
    rows.append((s.name, payload.get("_implementation", "(no provenance)"), ", ".join(payload.get("_objects_read", [])), (s.end_time_ns - s.start_time_ns) / 1e6))
print(f"{'tool span':<24} {'implementation':<42} {'ms':>8}  objects read")
print("-" * 150)
for name, impl, objs, ms in rows:
    print(f"{name:<24} {impl:<42} {ms:>8.1f}  {objs}")
assert rows, "no tool spans found"
assert all(r[1] == "agent_python:postgres:fraud_triage_agent" for r in rows), "a tool ran outside Postgres governance"
print(f"\n{len(rows)} tool spans, all Postgres-governed (fraud_triage_agent); no UC function tool was called.")

2026/11/01 19:07:45 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-8d32adad-a55b-4953-ba99-1953b044617b/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's... new one if you want.'}], input_type=list])
  return self.__pydantic_serializer__.to_python(


Latency: 16.0s
## TXN-0234209: $323.96, CUST-08669

**Why it was flagged**
The risk score is **0.984**. Three signals drive it:
- **New device (SHAP +2.57):** This is the strongest signal. The customer normally uses about 5 devices a month, so a new device is less unusual for them than for most customers.
- **Distance from home (SHAP +1.87):** The transaction was 450.6 km from the home region, US-East.
- **Online transaction (SHAP +1.20):** Online transactions score higher for risk.

**Recommended action:** **Block.** This is the model's recommendation.

**Confidence:** **Medium (observed precision 0.50 in block_band, n=288).** The score is very high, but historically only about half of the transactions in this band were confirmed fraud. Treat the score as a strong flag, not a near-certain one.

**Routed queue:** `auto_block`

**Customer context**
- The customer has a low risk segment, 40 months of tenure, and a US-East home region.
- In the last 30 days they made 12 transactions avera

Trace(trace_id='tr-372bc4fc3a27c07127bdc96756ac3920')

In [0]:
# E.2: Open a case
request_act = ResponsesAgentRequest(
    input=[{"role": "user", "content": "Open a case for TXN-0234209 and route it."}]
)
t0 = time.time()
response_act = agent.predict(request_act)
print(f"Latency: {time.time()-t0:.1f}s")
print(response_act.output[0].content[0]["text"])
# Case 2319d4b7-bea3-412b-ba6b-8325e1a7a71d (opened in Step 5) is still open, so the
# idempotent create_fraud_case should return that same case_id.
from lakebase_cases import LakebaseCases
_lc = LakebaseCases()
for c in _lc.get_cases_for_txn("TXN-0234209"):
    print(f"case {c['case_id']} status={c['status']} queue={c['assigned_queue']} created_by={c['created_by']} db_inserted_by={c['db_inserted_by']}")
_lc.close()

/local_disk0/.ephemeral_nfs/envs/pythonEnv-8d32adad-a55b-4953-ba99-1953b044617b/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...Wd1TRsHTFF3iV9J9xgB'}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-8d32adad-a55b-4953-ba99-1953b044617b/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...xisting one as it is?"}], input_type=list])
  return self.__pydantic_serializer__.to_python(


Latency: 10.6s
I haven't created a new case, because TXN-0234209 already has one open.

**Existing case:** `2319d4b7-bea3-412b-ba6b-8325e1a7a71d`, status **open**, queue **auto_block**. A second case would duplicate it.

## TXN-0234209: $323.96, CUST-08669

**Why flagged** (risk score 0.984)
- **New device (SHAP +2.57):** the customer normally uses 5 devices in 30 days.
- **450.6 km from home (SHAP +1.87):** the home region is US-East.
- **Online transaction (SHAP +1.20):** this channel carries higher risk.

**Recommended action:** block.

**Confidence:** medium (observed precision 0.50 in block_band, n=288). Only about half of transactions in this band turn out to be fraud, so a human should review the block.

**Routed queue:** auto_block.

**Customer context**
- Low-risk segment, 40 months of tenure, home region US-East.
- Prior history: 3 cases, all 3 confirmed fraud.
- In the last 30 days: 12 transactions averaging about $100, 5 devices, 4 countries and 5 cross-border transactions.

Trace(trace_id='tr-2e573295e00f2d579ae2cc85618c2abe')

In [0]:
# E.3: Ask again - should return same case_id
response_idem = agent.predict(ResponsesAgentRequest(
    input=[{"role": "user", "content": "Open a case for TXN-0234209 and route it."}]
))
print(response_idem.output[0].content[0]["text"][:1000])
from lakebase_cases import LakebaseCases
lc = LakebaseCases()
open_cases = [c for c in lc.get_cases_for_txn('TXN-0234209') if c['status'] != 'closed']
print(f"\nOpen cases after 2 requests: {len(open_cases)} (expected: 1)")
for c in open_cases:
    print(f"  case_id={c['case_id']}")
lc.close()

/local_disk0/.ephemeral_nfs/envs/pythonEnv-8d32adad-a55b-4953-ba99-1953b044617b/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...9uHfCJ1mdQn/UsyGAE='}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-8d32adad-a55b-4953-ba99-1953b044617b/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...del's action (block)."}], input_type=list])
  return self.__pydantic_serializer__.to_python(


I haven't opened a new case for TXN-0234209. An open case already exists for it, and I didn't want to create a duplicate.

**Existing case:** `2319d4b7-bea3-412b-ba6b-8325e1a7a71d`. Its status is open and it's in the `auto_block` queue.

## TXN-0234209: $323.96, CUST-08669
**Why flagged**
- **New device (SHAP +2.57):** The customer normally uses 5 devices in 30 days, and this one is new.
- **Distance from home (SHAP +1.87):** The transaction was 450.6 km from the home region, US-East.
- **Online transaction (SHAP +1.20):** Online transactions score as higher risk.

**Recommended action:** Block. The risk score is 0.984.

**Confidence:** Medium (observed precision 0.50 in block_band, n=288). Only about half of past transactions in this band were confirmed fraud, so the high score alone overstates how sure the model is.

**Routed queue:** `auto_block`

**Customer context**
- The customer is low-risk, with 40 months of tenure and a home region of US-East.
- They have 3 prior cases, and al

Trace(trace_id='tr-85307d847f0205ef7832b13288f36c02')

## E.5 Batch Triage - Top 5 Block Transactions
| txn_id | risk_score | case_id |
|--------|-----------|--------|
| TXN-0147293 | 0.993762 | 82b12d25-bf88-47cb-b0a4-f5a5b62f349d |
| TXN-0288343 | 0.992881 | 863334d8-909e-4348-8be4-772d0c7f6e72 |
| TXN-0221786 | 0.992875 | c37587c0-5091-4fd7-a656-c9df9417d0e3 |
| TXN-0417769 | 0.992698 | 98edced4-9dc7-47a3-bbf3-1694bb7b1b43 |
| TXN-0432087 | 0.992657 | 98adcae4-cec6-46a1-a2c5-1ecd84c299ad |

In [0]:
# E.6: Guardrails
# Unknown txn
resp_unknown = agent.predict(ResponsesAgentRequest(
    input=[{"role": "user", "content": "Why was TXN-9999999 flagged?"}]
))
print("--- Unknown txn ---")
print(resp_unknown.output[0].content[0]["text"][:500])

# Approve band should not be blocked
resp_approve = agent.predict(ResponsesAgentRequest(
    input=[{"role": "user", "content": "Block TXN-0278800 immediately."}]
))
print("\n--- Approve-band ---")
print(resp_approve.output[0].content[0]["text"][:500])

--- Unknown txn ---
I couldn't find **TXN-9999999**. Both the risk lookup and the signal-explanation lookup came back empty. That means I have no score, recommended action, confidence level, routed queue, or SHAP signals for it. I can't say why it was flagged, and I won't guess.

Here is what you could check:
- **Typo or format:** Confirm the ID is exactly right, including the prefix and digit count.
- **Scoring status:** The transaction may not have been scored yet, or it may be outside the current risk tables.
- 


/local_disk0/.ephemeral_nfs/envs/pythonEnv-8d32adad-a55b-4953-ba99-1953b044617b/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...is transaction first."}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-8d32adad-a55b-4953-ba99-1953b044617b/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...Which would you like?"}], input_type=list])
  return self.__pydantic_serializer__.to_python(



--- Approve-band ---
I haven't blocked TXN-0278800. The model scored it in the approve band, and I shouldn't push an approve-band transaction into block. Tell me how you want to proceed and I'll open the case.

**Why it was flagged**
- **New device (SHAP +1.76):** This is the strongest signal. The customer normally uses about 5 devices.
- **Online transaction (SHAP +0.99):** Online transactions carry higher risk.
- **Distance from home (SHAP +0.62):** The transaction was 575.6 km from the customer's EU home region.



[Trace(trace_id='tr-a88587ca4a26ed9111b7f2923c1966a0'), Trace(trace_id='tr-c03d76d0b0368e0599a8f3bbc07fdf8b')]

In [0]:
# E.7 (Step 7 Part E VERIFY): explain_top_signals previously raised
# "Unknown format code 'f' for object of type 'str'" for TXN-0112255 (a SHAP signal value was a string).
# Read-only: calls the tool directly against fraud_gov.* as fraud_triage_agent. Writes nothing.
from fraud_triage_agent import explain_top_signals
for txn in ["TXN-0112255", "TXN-0234209"]:
    out = json.loads(explain_top_signals.invoke({"txn_id": txn}))
    rows = out.get("rows") or out.get("content") or out
    print(f"=== {txn}: OK, implementation={out.get('_implementation')} ===")
    for r in (rows if isinstance(rows, list) else [rows]):
        print(" ", r)
print("\nexplain_top_signals returned for both transactions with no format error.")

=== TXN-0112255: OK, implementation=agent_python:postgres:fraud_triage_agent ===
  {'txn_id': 'TXN-0112255', 'signal_name': 'is_new_device', 'shap_value': 1.9395, 'feature_value': '1', 'customer_normal': 'Customer uses 5 devices in 30d', 'explanation': 'New device (SHAP +1.94). Normally uses 5 devices.'}
  {'txn_id': 'TXN-0112255', 'signal_name': 'geo_distance_from_home', 'shap_value': 1.8503, 'feature_value': '332.9 km', 'customer_normal': 'Home region: US-East', 'explanation': '332.9 km from home (US-East), SHAP +1.85.'}
  {'txn_id': 'TXN-0112255', 'signal_name': 'risk_segment', 'shap_value': 0.8605, 'feature_value': 'high', 'customer_normal': 'See customer profile', 'explanation': 'risk_segment = high (SHAP +0.86)'}
=== TXN-0234209: OK, implementation=agent_python:postgres:fraud_triage_agent ===
  {'txn_id': 'TXN-0234209', 'signal_name': 'is_new_device', 'shap_value': 2.5724, 'feature_value': '1', 'customer_normal': 'Customer uses 5 devices in 30d', 'explanation': 'New device (SHAP 

[Trace(trace_id='tr-e4020b760fc258d0b4ca95939c1f447b'), Trace(trace_id='tr-535fd86d5a7be72617009dbf84936fe6')]

In [0]:
# D.3: Log and register to UC
# GUARD (2026-10-21 hardening): v1 @champion was registered in Step 5 with the now-deprecated UC function
# tools. Re-registering creates v2 and moves @champion, so it runs only when explicitly enabled.
# Safety flag: False = no new agent version is logged or registered, @champion is not moved.
REGISTER_AGENT = False
from mlflow.tracking import MlflowClient
FULL_NAME = "workshop_guy_catalog.fe_bar_a0.fraud_triage_agent"
mc = MlflowClient(registry_uri="databricks-uc")

if not REGISTER_AGENT:
    v = mc.get_model_version_by_alias(FULL_NAME, "champion")
    print(f"Registration skipped (REGISTER_AGENT=False). Current @champion: v{v.version}, run {v.run_id}. It still embeds the pre-hardening UC-function tools.")
else:
    from mlflow.models.resources import DatabricksServingEndpoint
    resources = [
        DatabricksServingEndpoint(endpoint_name="databricks-claude-sonnet-5-5"),
    ]  # read tools are agent-side Postgres tools now; no UC function resources
    with mlflow.start_run(run_name="fraud_triage_agent_v1"):
        info = mlflow.pyfunc.log_model(
            name="agent",
            python_model="/Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo/fraud_triage_agent.py",
            resources=resources,
            input_example={"input": [{"role": "user", "content": "Why was TXN-0234209 flagged?"}]},
            pip_requirements=["mlflow>=3.1", "databricks-langchain", "langgraph>=0.3,<0.4",
                              "databricks-agents", "pydantic>=2", "psycopg[binary]>=3.1.0", "databricks-ai-search"],
            registered_model_name=FULL_NAME,
        )
    mc.set_registered_model_alias(FULL_NAME, "champion", info.registered_model_version)
    print(f"Registered: {FULL_NAME} v{info.registered_model_version} @champion")

Registration skipped (REGISTER_AGENT=False). Current @champion: v2, run 97bc6c9b4e654abab627cd25ee93fa53. It still embeds the pre-hardening UC-function tools.
